In [1]:
# ============================================================
# RANDOM FOREST - 5-FOLD CROSS VALIDATION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# STEP 1: LOAD CLEANED DATASET
# ============================================================

df = pd.read_csv(
    'phishing_website_cleaned.csv'
)

print("=" * 65)
print("RANDOM FOREST - 5-FOLD CROSS VALIDATION")
print("=" * 65)

print("\nDataset shape:")
print(df.shape)


# ============================================================
# STEP 2: DEFINE URL FEATURES
# ============================================================

URL_FEATURES = [

    # URL character features
    'qty_dot_url',
    'qty_hyphen_url',
    'qty_underline_url',
    'qty_slash_url',
    'qty_questionmark_url',
    'qty_equal_url',
    'qty_at_url',
    'qty_and_url',
    'qty_exclamation_url',
    'qty_space_url',
    'qty_tilde_url',
    'qty_comma_url',
    'qty_plus_url',
    'qty_asterisk_url',
    'qty_hashtag_url',
    'qty_dollar_url',
    'qty_percent_url',
    'length_url',

    # Domain character features
    'qty_dot_domain',
    'qty_hyphen_domain',
    'qty_underline_domain',
    'qty_slash_domain',
    'qty_questionmark_domain',
    'qty_equal_domain',
    'qty_at_domain',
    'qty_and_domain',
    'qty_exclamation_domain',
    'qty_space_domain',
    'qty_tilde_domain',
    'qty_comma_domain',
    'qty_plus_domain',
    'qty_asterisk_domain',
    'qty_hashtag_domain',
    'qty_dollar_domain',
    'qty_percent_domain',

    # Domain statistics
    'qty_vowels_domain',
    'domain_length',
    'domain_in_ip',
    'server_client_domain',
    'email_in_url',

    # Security / URL features
    'tls_ssl_certificate',
    'url_shortened'
]


# ============================================================
# STEP 3: CHECK AVAILABLE FEATURES
# ============================================================

available_features = [
    col
    for col in URL_FEATURES
    if col in df.columns
]

missing_features = [
    col
    for col in URL_FEATURES
    if col not in df.columns
]

print("\n" + "=" * 65)
print("FEATURE CHECK")
print("=" * 65)

print(
    "\nAvailable features:",
    len(available_features)
)

if missing_features:

    print("\nMissing features:")

    for col in missing_features:
        print("✗", col)

else:

    print("✓ All required features are available.")


# ============================================================
# STEP 4: CREATE X AND y
# ============================================================

X = df[
    available_features
].copy()

y = df[
    'phishing'
].copy()


print("\n" + "=" * 65)
print("FEATURE AND TARGET")
print("=" * 65)

print("\nFeatures shape:")
print(X.shape)

print("\nTarget shape:")
print(y.shape)

print("\nTarget distribution:")
print(y.value_counts())


# ============================================================
# STEP 5: REMOVE CONSTANT FEATURES
# ============================================================

constant_features = [
    col
    for col in X.columns
    if X[col].nunique() <= 1
]

print("\n" + "=" * 65)
print("CONSTANT FEATURE CHECK")
print("=" * 65)

print(
    "\nConstant features found:",
    len(constant_features)
)

if constant_features:

    print("\nRemoving constant features:")

    for col in constant_features:
        print("✗", col)

    X = X.drop(
        columns=constant_features
    )

else:

    print("✓ No constant features found.")


print(
    "\nFeatures used for cross validation:",
    X.shape[1]
)


# ============================================================
# STEP 6: CREATE RANDOM FOREST MODEL
# ============================================================

model = RandomForestClassifier(

    n_estimators=100,

    random_state=42,

    n_jobs=-1

)


# ============================================================
# STEP 7: CREATE 5-FOLD STRATIFIED CROSS VALIDATION
# ============================================================

cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)


print("\n" + "=" * 65)
print("5-FOLD CROSS VALIDATION")
print("=" * 65)

print("\nNumber of folds:", cv.n_splits)

print("Shuffle:", True)

print("Random state:", 42)


# ============================================================
# STEP 8: DEFINE EVALUATION METRICS
# ============================================================

scoring = {

    'accuracy': 'accuracy',

    'precision': 'precision',

    'recall': 'recall',

    'f1': 'f1'

}


# ============================================================
# STEP 9: RUN CROSS VALIDATION
# ============================================================

print("\nRunning 5-Fold Cross Validation...")

cv_results = cross_validate(

    model,

    X,

    y,

    cv=cv,

    scoring=scoring,

    n_jobs=-1

)


print("\n✓ Cross validation completed successfully.")


# ============================================================
# STEP 10: DISPLAY RESULTS FOR EACH FOLD
# ============================================================

print("\n" + "=" * 65)
print("RESULTS FOR EACH FOLD")
print("=" * 65)

for i in range(5):

    print(
        f"\nFold {i + 1}:"
    )

    print(
        f"  Accuracy : "
        f"{cv_results['test_accuracy'][i] * 100:.2f}%"
    )

    print(
        f"  Precision: "
        f"{cv_results['test_precision'][i] * 100:.2f}%"
    )

    print(
        f"  Recall   : "
        f"{cv_results['test_recall'][i] * 100:.2f}%"
    )

    print(
        f"  F1-Score : "
        f"{cv_results['test_f1'][i] * 100:.2f}%"
    )


# ============================================================
# STEP 11: CALCULATE AVERAGE PERFORMANCE
# ============================================================

accuracy_mean = cv_results[
    'test_accuracy'
].mean()

precision_mean = cv_results[
    'test_precision'
].mean()

recall_mean = cv_results[
    'test_recall'
].mean()

f1_mean = cv_results[
    'test_f1'
].mean()


accuracy_std = cv_results[
    'test_accuracy'
].std()

precision_std = cv_results[
    'test_precision'
].std()

recall_std = cv_results[
    'test_recall'
].std()

f1_std = cv_results[
    'test_f1'
].std()


# ============================================================
# STEP 12: FINAL CROSS VALIDATION RESULTS
# ============================================================

print("\n" + "=" * 65)
print("FINAL 5-FOLD CROSS VALIDATION RESULTS")
print("=" * 65)

print(
    f"\nAccuracy : "
    f"{accuracy_mean * 100:.2f}% "
    f"(± {accuracy_std * 100:.2f}%)"
)

print(
    f"Precision: "
    f"{precision_mean * 100:.2f}% "
    f"(± {precision_std * 100:.2f}%)"
)

print(
    f"Recall   : "
    f"{recall_mean * 100:.2f}% "
    f"(± {recall_std * 100:.2f}%)"
)

print(
    f"F1-Score : "
    f"{f1_mean * 100:.2f}% "
    f"(± {f1_std * 100:.2f}%)"
)


# ============================================================
# STEP 13: SUMMARY TABLE
# ============================================================

results = pd.DataFrame({

    'Metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1-Score'
    ],

    'Mean': [

        accuracy_mean * 100,

        precision_mean * 100,

        recall_mean * 100,

        f1_mean * 100

    ],

    'Std': [

        accuracy_std * 100,

        precision_std * 100,

        recall_std * 100,

        f1_std * 100

    ]

})


print("\n" + "=" * 65)
print("SUMMARY TABLE")
print("=" * 65)

print(
    results.round(2).to_string(
        index=False
    )
)


# ============================================================
# STEP 14: TRAIN FINAL RANDOM FOREST MODEL
# ============================================================

print("\n" + "=" * 65)
print("FINAL RANDOM FOREST MODEL")
print("=" * 65)

final_model = RandomForestClassifier(

    n_estimators=100,

    random_state=42,

    n_jobs=-1

)

final_model.fit(
    X,
    y
)

print(
    "\n✓ Final Random Forest model trained "
    "using the complete dataset."
)


# ============================================================
# END
# ============================================================

print("\n" + "=" * 65)
print("5-FOLD CROSS VALIDATION COMPLETED")
print("=" * 65)

RANDOM FOREST - 5-FOLD CROSS VALIDATION

Dataset shape:
(9944, 56)

FEATURE CHECK

Available features: 42
✓ All required features are available.

FEATURE AND TARGET

Features shape:
(9944, 42)

Target shape:
(9944,)

Target distribution:
phishing
0    6491
1    3453
Name: count, dtype: int64

CONSTANT FEATURE CHECK

Constant features found: 15

Removing constant features:
✗ qty_hashtag_url
✗ qty_slash_domain
✗ qty_questionmark_domain
✗ qty_equal_domain
✗ qty_at_domain
✗ qty_and_domain
✗ qty_exclamation_domain
✗ qty_space_domain
✗ qty_tilde_domain
✗ qty_comma_domain
✗ qty_plus_domain
✗ qty_asterisk_domain
✗ qty_hashtag_domain
✗ qty_dollar_domain
✗ qty_percent_domain

Features used for cross validation: 27

5-FOLD CROSS VALIDATION

Number of folds: 5
Shuffle: True
Random state: 42

Running 5-Fold Cross Validation...

✓ Cross validation completed successfully.

RESULTS FOR EACH FOLD

Fold 1:
  Accuracy : 92.16%
  Precision: 88.60%
  Recall   : 88.86%
  F1-Score : 88.73%

Fold 2:
  Accurac

In [2]:
# ============================================================
# RANDOM FOREST Model set up for PHISHING URL DETECTION
# ============================================================

import pandas as pd
import numpy as np
import re

from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# STEP 1: LOAD CLEANED DATASET
# ============================================================

df = pd.read_csv(
    'phishing_website_cleaned.csv'
)

print("=" * 60)
print("RANDOM FOREST PHISHING URL DETECTION")
print("=" * 60)

print("\nDataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# STEP 2: DEFINE URL FEATURES
# ============================================================

URL_FEATURES = [

    # -------------------------
    # URL character features
    # -------------------------

    'qty_dot_url',
    'qty_hyphen_url',
    'qty_underline_url',
    'qty_slash_url',
    'qty_questionmark_url',
    'qty_equal_url',
    'qty_at_url',
    'qty_and_url',
    'qty_exclamation_url',
    'qty_space_url',
    'qty_tilde_url',
    'qty_comma_url',
    'qty_plus_url',
    'qty_asterisk_url',
    'qty_hashtag_url',
    'qty_dollar_url',
    'qty_percent_url',
    'length_url',

    # -------------------------
    # Domain character features
    # -------------------------

    'qty_dot_domain',
    'qty_hyphen_domain',
    'qty_underline_domain',
    'qty_slash_domain',
    'qty_questionmark_domain',
    'qty_equal_domain',
    'qty_at_domain',
    'qty_and_domain',
    'qty_exclamation_domain',
    'qty_space_domain',
    'qty_tilde_domain',
    'qty_comma_domain',
    'qty_plus_domain',
    'qty_asterisk_domain',
    'qty_hashtag_domain',
    'qty_dollar_domain',
    'qty_percent_domain',

    # -------------------------
    # Domain statistics
    # -------------------------

    'qty_vowels_domain',
    'domain_length',
    'domain_in_ip',
    'server_client_domain',
    'email_in_url',

    # -------------------------
    # Security / URL features
    # -------------------------

    'tls_ssl_certificate',
    'url_shortened'
]


# ============================================================
# STEP 3: CHECK AVAILABLE FEATURES
# ============================================================

available_features = [

    col
    for col in URL_FEATURES
    if col in df.columns

]

missing_features = [

    col
    for col in URL_FEATURES
    if col not in df.columns

]


print("\n" + "=" * 60)
print("FEATURE CHECK")
print("=" * 60)

print(
    "\nNumber of available URL features:",
    len(available_features)
)


print("\nAvailable features:")

for col in available_features:

    print("✓", col)


if missing_features:

    print("\nMissing features:")

    for col in missing_features:

        print("✗", col)


# ============================================================
# STEP 4: CREATE X AND y
# ============================================================

X = df[
    available_features
].copy()


y = df[
    'phishing'
].copy()


print("\n" + "=" * 60)
print("FEATURE AND TARGET")
print("=" * 60)

print(
    "\nFeatures shape:",
    X.shape
)

print(
    "Target shape:",
    y.shape
)


print("\nTarget distribution:")

print(
    y.value_counts()
)


# ============================================================
# STEP 5: REMOVE CONSTANT FEATURES
# ============================================================

constant_features = [

    col
    for col in X.columns
    if X[col].nunique() <= 1

]


print("\n" + "=" * 60)
print("CONSTANT FEATURE CHECK")
print("=" * 60)

print(
    "\nNumber of constant features:",
    len(constant_features)
)


if constant_features:

    print("\nRemoving constant features:")

    for col in constant_features:

        print("-", col)


    X = X.drop(
        columns=constant_features
    )


print(
    "\nFeatures after removing constant features:",
    X.shape[1]
)


# ============================================================
# STEP 6: TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)


print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print(
    "\nTraining data:",
    X_train.shape
)

print(
    "Testing data:",
    X_test.shape
)


print("\nTraining target distribution:")

print(
    y_train.value_counts()
)


print("\nTesting target distribution:")

print(
    y_test.value_counts()
)


# ============================================================
# STEP 7: STANDARDIZATION
# ============================================================

scaler = StandardScaler()


# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(
    X_train
)


# Transform ONLY test data
X_test_scaled = scaler.transform(
    X_test
)


print("\n" + "=" * 60)
print("STANDARDIZATION")
print("=" * 60)

print("\n✓ Standardization completed")

print(
    "\nTraining scaled shape:",
    X_train_scaled.shape
)

print(
    "Testing scaled shape:",
    X_test_scaled.shape
)


# ============================================================
# STEP 8: RANDOM FOREST MODEL
# ============================================================

from sklearn.ensemble import RandomForestClassifier


model = RandomForestClassifier(

    n_estimators=100,

    random_state=42,

    n_jobs=-1

)


model.fit(

    X_train_scaled,

    y_train

)


print("\n" + "=" * 60)
print("RANDOM FOREST TRAINING")
print("=" * 60)

print(
    "\n✓ Random Forest training completed"
)


# ============================================================
# STEP 9: PREDICTION ON TEST DATA
# ============================================================

y_pred = model.predict(

    X_test_scaled

)


# ============================================================
# STEP 10: MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(

    y_test,

    y_pred

)


print("\n" + "=" * 60)
print("MODEL EVALUATION")
print("=" * 60)


print(
    "\nAccuracy:",
    round(
        accuracy * 100,
        2
    ),
    "%"
)


print("\nClassification Report:")

print(

    classification_report(

        y_test,

        y_pred,

        target_names=[
            "Legitimate",
            "Phishing"
        ]

    )

)


print("\nConfusion Matrix:")

cm = confusion_matrix(

    y_test,

    y_pred

)

print(cm)


# ============================================================
# STEP 11: EXTRACT FEATURES FROM NEW URL
# ============================================================

def extract_url_features(url):

    # --------------------------------------------------------
    # Clean URL
    # --------------------------------------------------------

    url = url.strip()


    # --------------------------------------------------------
    # Handle Markdown-style URL
    # --------------------------------------------------------

    if url.startswith('[') and '](' in url:

        url = url.split(
            '](',
            1
        )[1]


        if url.endswith(')'):

            url = url[:-1]


    url = url.strip()


    # --------------------------------------------------------
    # Add HTTP scheme if missing
    # --------------------------------------------------------

    if not url.startswith(
        ('http://', 'https://')
    ):

        url = 'http://' + url


    # --------------------------------------------------------
    # Parse URL
    # --------------------------------------------------------

    parsed = urlparse(url)

    full_url = url

    domain = parsed.netloc.split(':')[0]


    features = {}


    # ========================================================
    # URL CHARACTER FEATURES
    # ========================================================

    features['qty_dot_url'] = \
        full_url.count('.')

    features['qty_hyphen_url'] = \
        full_url.count('-')

    features['qty_underline_url'] = \
        full_url.count('_')

    features['qty_slash_url'] = \
        full_url.count('/')

    features['qty_questionmark_url'] = \
        full_url.count('?')

    features['qty_equal_url'] = \
        full_url.count('=')

    features['qty_at_url'] = \
        full_url.count('@')

    features['qty_and_url'] = \
        full_url.count('&')

    features['qty_exclamation_url'] = \
        full_url.count('!')

    features['qty_space_url'] = \
        full_url.count(' ')

    features['qty_tilde_url'] = \
        full_url.count('~')

    features['qty_comma_url'] = \
        full_url.count(',')

    features['qty_plus_url'] = \
        full_url.count('+')

    features['qty_asterisk_url'] = \
        full_url.count('*')

    features['qty_hashtag_url'] = \
        full_url.count('#')

    features['qty_dollar_url'] = \
        full_url.count('$')

    features['qty_percent_url'] = \
        full_url.count('%')


    # ========================================================
    # URL LENGTH
    # ========================================================

    features['length_url'] = \
        len(full_url)


    # ========================================================
    # DOMAIN CHARACTER FEATURES
    # ========================================================

    features['qty_dot_domain'] = \
        domain.count('.')

    features['qty_hyphen_domain'] = \
        domain.count('-')

    features['qty_underline_domain'] = \
        domain.count('_')

    features['qty_slash_domain'] = \
        domain.count('/')

    features['qty_questionmark_domain'] = \
        domain.count('?')

    features['qty_equal_domain'] = \
        domain.count('=')

    features['qty_at_domain'] = \
        domain.count('@')

    features['qty_and_domain'] = \
        domain.count('&')

    features['qty_exclamation_domain'] = \
        domain.count('!')

    features['qty_space_domain'] = \
        domain.count(' ')

    features['qty_tilde_domain'] = \
        domain.count('~')

    features['qty_comma_domain'] = \
        domain.count(',')

    features['qty_plus_domain'] = \
        domain.count('+')

    features['qty_asterisk_domain'] = \
        domain.count('*')

    features['qty_hashtag_domain'] = \
        domain.count('#')

    features['qty_dollar_domain'] = \
        domain.count('$')

    features['qty_percent_domain'] = \
        domain.count('%')


    # ========================================================
    # DOMAIN STATISTICS
    # ========================================================

    features['qty_vowels_domain'] = sum(

        c.lower() in 'aeiou'

        for c in domain

    )


    features['domain_length'] = \
        len(domain)


    # ========================================================
    # DOMAIN IN IP ADDRESS
    # ========================================================

    ip_pattern = \
        r'^(\d{1,3}\.){3}\d{1,3}$'


    features['domain_in_ip'] = int(

        bool(

            re.match(

                ip_pattern,

                domain

            )

        )

    )


    # ========================================================
    # SERVER / CLIENT DOMAIN
    # ========================================================

    features['server_client_domain'] = int(

        'server' in domain.lower()

        or

        'client' in domain.lower()

    )


    # ========================================================
    # EMAIL IN URL
    # ========================================================

    features['email_in_url'] = int(

        '@' in full_url

    )


    # ========================================================
    # HTTPS / SSL
    # ========================================================

    features['tls_ssl_certificate'] = int(

        parsed.scheme == 'https'

    )


    # ========================================================
    # URL SHORTENER
    # ========================================================

    shorteners = [

        'bit.ly',
        'tinyurl.com',
        't.co',
        'goo.gl',
        'is.gd',
        'ow.ly'

    ]


    features['url_shortened'] = int(

        any(

            shortener in domain.lower()

            for shortener in shorteners

        )

    )


    return features


# ============================================================
# STEP 12: PREPARE URL FOR MODEL
# ============================================================

def prepare_url_for_model(url):

    # Extract features
    features = extract_url_features(
        url
    )


    # Convert to DataFrame
    url_df = pd.DataFrame(
        [features]
    )


    # --------------------------------------------------------
    # Check for missing features
    # --------------------------------------------------------

    missing_features = [

        col

        for col in X.columns

        if col not in url_df.columns

    ]


    if missing_features:

        raise ValueError(

            "Missing features for prediction: "

            + str(missing_features)

        )


    # --------------------------------------------------------
    # Use EXACT same feature order
    # --------------------------------------------------------

    url_df = url_df[
        X.columns
    ]


    # --------------------------------------------------------
    # Apply the same scaler
    # --------------------------------------------------------

    url_scaled = scaler.transform(
        url_df
    )


    return url_scaled


# ============================================================
# STEP 13: PREDICT NEW URL
# ============================================================

def predict_url(url):

    # Prepare URL
    url_scaled = prepare_url_for_model(
        url
    )


    # Make prediction
    prediction = model.predict(
        url_scaled
    )[0]


    # Get probability
    probabilities = model.predict_proba(
        url_scaled
    )[0]


    legitimate_probability = \
        probabilities[0]

    phishing_probability = \
        probabilities[1]


    print("\n" + "=" * 55)
    print("URL PHISHING DETECTION")
    print("=" * 55)


    print("\nURL:")

    print(url)


    print(

        f"\nLegitimate probability: "
        f"{legitimate_probability * 100:.2f}%"

    )


    print(

        f"Phishing probability: "
        f"{phishing_probability * 100:.2f}%"

    )


    if prediction == 1:

        print(
            "\nResult: ⚠️ PHISHING URL"
        )

    else:

        print(
            "\nResult: ✅ LEGITIMATE URL"
        )


    print("=" * 55)


    return prediction


# ============================================================
# END OF MODEL RANDOM FOREST
# ============================================================

RANDOM FOREST PHISHING URL DETECTION

Dataset shape:
(9944, 56)

Columns:
['qty_dot_url', 'qty_hyphen_url', 'qty_underline_url', 'qty_slash_url', 'qty_questionmark_url', 'qty_equal_url', 'qty_at_url', 'qty_and_url', 'qty_exclamation_url', 'qty_space_url', 'qty_tilde_url', 'qty_comma_url', 'qty_plus_url', 'qty_asterisk_url', 'qty_hashtag_url', 'qty_dollar_url', 'qty_percent_url', 'qty_tld_url', 'length_url', 'qty_dot_domain', 'qty_hyphen_domain', 'qty_underline_domain', 'qty_slash_domain', 'qty_questionmark_domain', 'qty_equal_domain', 'qty_at_domain', 'qty_and_domain', 'qty_exclamation_domain', 'qty_space_domain', 'qty_tilde_domain', 'qty_comma_domain', 'qty_plus_domain', 'qty_asterisk_domain', 'qty_hashtag_domain', 'qty_dollar_domain', 'qty_percent_domain', 'qty_vowels_domain', 'domain_length', 'domain_in_ip', 'server_client_domain', 'email_in_url', 'time_response', 'domain_spf', 'asn_ip', 'time_domain_activation', 'time_domain_expiration', 'qty_ip_resolved', 'qty_nameservers', 'qty_m